# Final Project: Multi-LLM Collaborative Debate System

**Course:** Applied LLM Systems

The idea of this system is to fight hallucinations and reasoning errors through debate between multiple LLMs.
Instead of trusting a single model, several models solve the same problem, criticize each other,
fix their solutions based on the feedback, and finally an independent judge picks the best answer.

The pipeline has these stages:

- **Stage 0** - Role self-assessment: every agent looks at the problem and says how confident it would be as a Solver vs as a Judge
- **Stage 0.5** - Deterministic role assignment: a simple algorithm turns those self-assessments into 3 Solvers + 1 Judge
- **Stage 1** - The 3 Solvers solve the problem independently (no communication between them)
- **Stage 2** - Peer review: each Solver reviews the other two solutions with structured feedback
- **Stage 3** - Refinement: each Solver answers every critique (accept and fix, or reject and defend) and produces a refined solution
- **Stage 4** - The Judge sees everything (originals, reviews, refined solutions), verifies the reasoning and picks a winner

The winner's refined answer is the system's final answer.

## The 4 agents - 2 providers, strong + weak model from each

The agents are 4 different models: from each provider I take one strong modern model and one
deliberately weaker/older model:

| Agent | Provider | Model | Tier |
|---|---|---|---|
| openai | OpenAI | gpt-4.1-mini | mid (2025) |
| openai-35 | OpenAI | gpt-3.5-turbo | weak (2023) |
| claude | Anthropic | claude-sonnet-4-5 | strong |
| claude-haiku | Anthropic | claude-haiku-4-5 | small/fast |

The weak models are a DELIBERATE choice, not a budget one. In an early version all four agents
were frontier-tier models and every metric saturated at ~100% accuracy - you could not see the
debate doing anything. With a strong/weak mix the interesting dynamics become measurable: weak
solvers produce real errors, peer review has something to catch, refinement has something to
fix, and the judge has real disagreements to resolve.

(Historical note: the original lineup was OpenAI + Anthropic + Google Gemini + Llama-on-Groq,
i.e. 4 separate providers. The Gemini and Groq free-tier keys kept dying mid-run - exhausted
daily quotas and rate limits - so both slots were replaced with models from the two reliable
paid providers. The assignment explicitly allows using one provider with different models.)

All agents are called through the same OpenAI Python SDK: Anthropic exposes an OpenAI-compatible
endpoint, so the only per-agent configuration is `base_url` + key + model name. gpt-3.5-turbo
predates structured outputs (no json_schema support), so it runs through the fallback path in
`ask()`: plain JSON mode + manual Pydantic validation.

Which agent becomes the Judge is decided per problem in Stage 0.5.

**Note:** running this notebook end to end makes a lot of API calls (around 25-30 per problem, 25 problems).
Results are checkpointed to `results/debate_results.json` after every problem, so if it crashes you can just
re-run the cell and it continues where it stopped.

In [33]:
import json
import os
import time
from typing import List, Literal
from concurrent.futures import ThreadPoolExecutor

from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads all 4 API keys from the .env file (never hardcode keys!)

# one config entry per agent: which endpoint, which key, which model,
# and whether the endpoint supports native structured outputs (json_schema)
AGENTS = {
    "openai": {
        "model": "gpt-4.1-mini",
        "base_url": None,  # default OpenAI endpoint
        "api_key": os.environ["OPENAI_API_KEY"],
        "structured": True,
    },
    "claude": {
        "model": "claude-sonnet-4-5",
        "base_url": "https://api.anthropic.com/v1/",
        "api_key": os.environ["ANTHROPIC_API_KEY"],
        "structured": True,
    },
    "openai-35": {
        # deliberately an OLD model (2023 era): a weak solver makes the debate mechanism visible.
        # it also has no json_schema support, so it exercises the fallback path in ask()
        "model": "gpt-3.5-turbo",
        "base_url": None,
        "api_key": os.environ["OPENAI_API_KEY"],
        "structured": False,
    },
    "claude-haiku": {
        "model": "claude-haiku-4-5",
        "base_url": "https://api.anthropic.com/v1/",
        "api_key": os.environ["ANTHROPIC_API_KEY"],
        "structured": True,
    },
}

# one OpenAI SDK client per provider.
# timeout + max_retries=0 matter: the SDK default is a 10 MINUTE timeout with internal retries,
# so one unresponsive provider can silently stall the whole run for an hour. Our own retry loop
# in ask() handles retries with proper backoff instead.
clients = {
    name: OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"], timeout=120, max_retries=0)
    for name, cfg in AGENTS.items()
}

# separate model used only for grading answers against ground truth (it is not part of the debate)
GRADER_MODEL = "gpt-4.1-mini"
grader_client = clients["openai"]

RESULTS_PATH = "results/debate_results.json"
BASELINE_PATH = "results/baseline_results.json"
os.makedirs("results", exist_ok=True)

AGENT_NAMES = list(AGENTS.keys())
print("agents:", {n: AGENTS[n]["model"] for n in AGENT_NAMES})

agents: {'openai': 'gpt-4.1-mini', 'claude': 'claude-sonnet-4-5', 'openai-35': 'gpt-3.5-turbo', 'claude-haiku': 'claude-haiku-4-5'}


## Phase 1: Problem dataset

The dataset lives in `problems.json`: 25 problems in 5 categories. I deliberately did NOT use
the categories suggested in the assignment (those are what most projects will use). The design
goals for the problems:

1. **multi-step derivations, not lookups** - every problem requires an actual computation with
   the given numbers (a payment formula, an energy balance, a characteristic equation), so a
   model cannot answer from memorized trivia; it has to carry the arithmetic through correctly
2. **multi-part answers** - most problems ask for 2-3 values (e.g. radius AND height AND area),
   which makes partially-right-but-wrong answers common and gives the peer reviewers something
   concrete to catch
3. **verifiable ground truth** - every answer was computed and verified with Python before
   being added to the dataset (the verification script computes each value independently)

The 5 categories, exam-style:

- **optimization** - calculus optimization and related rates (minimal can surface, fence along
  a river, open-top box, related-rates ladder)
- **probability_expectation** - expected values that need state equations or clever
  decompositions (HH vs HT waiting times, gambler's ruin, coupon collector, first ace position)
- **applied_physics** - multi-step numeric derivations (incline with friction, projectile from
  a cliff, spring launch + energy conservation, RC circuit charging)
- **finance** - annuity and discounting formulas with exact numbers (mortgage payment, NPV,
  doubling time, future value of monthly savings, effective annual rates)
- **discrete_math** - CRT system, Fermat's little theorem, Euler's totient, solving a linear
  recurrence via the characteristic equation, base-7 arithmetic with carries

In [34]:
with open("problems.json", "r", encoding="utf-8") as f:
    problems = json.load(f)

print(f"loaded {len(problems)} problems")

from collections import Counter
print(Counter(p["category"] for p in problems))

print("\nexample problem:")
print(problems[0]["question"])
print("ground truth:", problems[0]["answer"])

loaded 25 problems
Counter({'optimization': 5, 'probability_expectation': 5, 'applied_physics': 5, 'finance': 5, 'discrete_math': 5})

example problem:
A closed cylindrical can (top and bottom included) must hold exactly 500 cm^3. Using calculus, find the radius and height that minimize the total surface area, and compute that minimal surface area. Give radius, height and area numerically (2 decimal places), and state the relationship between h and r at the optimum.
ground truth: r approx 4.30 cm, h approx 8.60 cm, minimal surface area approx 348.73 cm^2; at the optimum h = 2r (height equals diameter)
